# Day1: Genesis on Colab — Franka を置いて mp4 を1本出す

**ゴール**: `day1_franka.mp4`（Frankaアームが動く動画）が出力されること。

**事前準備**: メニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択（L4でも可。A100は不要）。

**注意**: `gs.init()` は1ランタイムで1回だけ。やり直すときは「ランタイム」→「セッションを再起動」してから上から実行。

In [ ]:
# 1. GPU と Python のバージョン確認（Genesis は Python 3.10〜3.13 対応）
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Genesis のインストール（Colab には CUDA 版 PyTorch が入っているので Genesis だけ入れる）
!pip install -q genesis-world
from importlib.metadata import version
print('genesis-world', version('genesis-world'))
# ↑ 表示されたバージョンを requirements.txt に genesis-world==X.Y.Z として固定すること

In [ ]:
# 3. シーン構築: 床 + Franka + キューブ + 録画用カメラ（画面なし）
import numpy as np
import genesis as gs

gs.init(backend=gs.gpu, logging_level='warning')

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=0.01),
    show_viewer=False,  # Colab では必ず False
)

plane = scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(
    gs.morphs.MJCF(file='xml/franka_emika_panda/panda.xml'),  # Genesis 同梱モデル
)
cube = scene.add_entity(
    gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=(0.65, 0.0, 0.02)),
)

cam = scene.add_camera(
    res=(1280, 720),
    pos=(2.0, -1.6, 1.4),
    lookat=(0.3, 0.0, 0.3),
    fov=40,
    GUI=False,
)

scene.build()  # 初回はカーネルのコンパイルで数分かかる

In [ ]:
# 4. 関節インデックスと PD ゲイン（公式チュートリアルの値）
joints_name = ('joint1','joint2','joint3','joint4','joint5','joint6','joint7',
               'finger_joint1','finger_joint2')
dofs = [franka.get_joint(n).dofs_idx_local[0] for n in joints_name]

franka.set_dofs_kp(np.array([4500,4500,3500,3500,2000,2000,2000,100,100]), dofs_idx_local=dofs)
franka.set_dofs_kv(np.array([450,450,350,350,200,200,200,10,10]), dofs_idx_local=dofs)
franka.set_dofs_force_range(
    lower=np.array([-87,-87,-87,-87,-12,-12,-12,-100,-100]),
    upper=np.array([ 87, 87, 87, 87, 12, 12, 12, 100, 100]),
    dofs_idx_local=dofs,
)

# 初期姿勢（set_* は物理を介さず瞬時に置く。リセット専用）
HOME = np.array([0.0, -0.785, 0.0, -2.356, 0.0, 1.571, 0.785, 0.04, 0.04])
POSE_B = np.array([0.8, -0.3, 0.0, -2.0, 0.0, 1.8, 0.785, 0.04, 0.04])
franka.set_dofs_position(HOME, dofs)

In [ ]:
# 5. 録画: HOME → POSE_B → HOME を PD 制御で往復（約4秒）
cam.start_recording(save_to_filename='day1_franka.mp4', fps=30)

for i in range(400):
    if i == 50:
        franka.control_dofs_position(POSE_B, dofs)
    elif i == 250:
        franka.control_dofs_position(HOME, dofs)
    elif i == 0:
        franka.control_dofs_position(HOME, dofs)
    scene.step()  # カメラは step 内で自動的にフレームを書き出す

cam.stop_recording()
print('saved: day1_franka.mp4')

In [ ]:
# 6. その場で再生して確認
from IPython.display import Video
Video('day1_franka.mp4', embed=True, width=720)

In [ ]:
# 7. Google Drive に退避（Colab のストレージはセッション終了で消える）
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pai_final
!cp day1_franka.mp4 /content/drive/MyDrive/pai_final/
!pip freeze | grep -i -E '^(genesis-world|torch)==' | tee /content/drive/MyDrive/pai_final/requirements_day1.txt

## 描画で失敗したとき（セル3〜5で OpenGL / EGL 関連のエラー）

「セッションを再起動」した後、**セル1の前に**次を実行してから上から流し直す。

```python
import os
os.environ['PYOPENGL_PLATFORM'] = 'egl'
```

それでもダメなら、ソフトウェア描画（遅いが確実）に切り替える：

```python
!apt-get -qq install -y libosmesa6-dev
import os
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
```

**撤退ライン**: 30分ハマったら今夜は打ち切り。火曜は MuJoCo に切り替える。

**コミット前**: 「編集」→「出力をすべて消去」。埋め込み動画やパスが残るため。